# CIC-IDS-2017 - Cross-File Data Quality Exploration

This notebook loops through all CSV files in the TrafficLabelling directory and performs:
1. Class distribution across all files (Label field)
2. Duplicate detection
3. Missing data analysis
4. Skewness analysis on numeric fields
5. Outlier detection on numeric fields
6. Correlation analysis on numeric fields

In [1]:
import pandas as pd
import numpy as np
import os
import dotenv
import warnings

warnings.filterwarnings('ignore')
dotenv.load_dotenv()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [2]:
data_dir = os.getenv('DATA_DIR', './data')
traffic_dir = os.path.join(data_dir, 'GeneratedLabelledFlows', 'TrafficLabelling')

csv_files = sorted([f for f in os.listdir(traffic_dir) if f.endswith('.csv')])
print(f'Found {len(csv_files)} CSV files in {traffic_dir}:\n')
for f in csv_files:
    print(f'  {f}')

Found 8 CSV files in /mnt/data/capstone/GeneratedLabelledFlows/TrafficLabelling:

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Morning.pcap_ISCX.csv
  Monday-WorkingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv


## Load All Files

In [3]:
dfs = {}
for filename in csv_files:
    filepath = os.path.join(traffic_dir, filename)
    df = pd.read_csv(filepath, encoding='cp1252')
    df.columns = df.columns.str.strip()
    dfs[filename] = df
    print(f'{filename}: {df.shape[0]:,} rows, {df.shape[1]} columns')

total_rows = sum(df.shape[0] for df in dfs.values())
print(f'\nTotal rows across all files: {total_rows:,}')

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows, 85 columns
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows, 85 columns
Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows, 85 columns
Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows, 85 columns
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows, 85 columns
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 458,968 rows, 85 columns
Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 rows, 85 columns
Wednesday-workingHours.pcap_ISCX.csv: 692,703 rows, 85 columns

Total rows across all files: 3,119,345


In [4]:
print(f'Column dtypes for {csv_files[-1]}:\n')
df.dtypes

Column dtypes for Wednesday-workingHours.pcap_ISCX.csv:



Flow ID                            str
Source IP                          str
Source Port                      int64
Destination IP                     str
Destination Port                 int64
Protocol                         int64
Timestamp                          str
Flow Duration                    int64
Total Fwd Packets                int64
Total Backward Packets           int64
Total Length of Fwd Packets      int64
Total Length of Bwd Packets    float64
Fwd Packet Length Max            int64
Fwd Packet Length Min            int64
Fwd Packet Length Mean         float64
Fwd Packet Length Std          float64
Bwd Packet Length Max            int64
Bwd Packet Length Min            int64
Bwd Packet Length Mean         float64
Bwd Packet Length Std          float64
Flow Bytes/s                   float64
Flow Packets/s                 float64
Flow IAT Mean                  float64
Flow IAT Std                   float64
Flow IAT Max                   float64
Flow IAT Min             

# 1. Class Distribution (Label Field)

## Per-File Label Counts

In [5]:
per_file_labels = {}
for filename, df in dfs.items():
    counts = df['Label'].value_counts()
    per_file_labels[filename] = counts
    short_name = filename.replace('.pcap_ISCX.csv', '')
    print(f'\n--- {short_name} ---')
    for label, count in counts.items():
        print(f'  {label}: {count:,}')


--- Friday-WorkingHours-Afternoon-DDos ---
  DDoS: 128,027
  BENIGN: 97,718

--- Friday-WorkingHours-Afternoon-PortScan ---
  PortScan: 158,930
  BENIGN: 127,537

--- Friday-WorkingHours-Morning ---
  BENIGN: 189,067
  Bot: 1,966

--- Monday-WorkingHours ---
  BENIGN: 529,918

--- Thursday-WorkingHours-Afternoon-Infilteration ---
  BENIGN: 288,566
  Infiltration: 36

--- Thursday-WorkingHours-Morning-WebAttacks ---
  BENIGN: 168,186
  Web Attack – Brute Force: 1,507
  Web Attack – XSS: 652
  Web Attack – Sql Injection: 21

--- Tuesday-WorkingHours ---
  BENIGN: 432,074
  FTP-Patator: 7,938
  SSH-Patator: 5,897

--- Wednesday-workingHours ---
  BENIGN: 440,031
  DoS Hulk: 231,073
  DoS GoldenEye: 10,293
  DoS slowloris: 5,796
  DoS Slowhttptest: 5,499
  Heartbleed: 11


## Aggregated Label Counts Across All Files

In [6]:
all_label_counts = pd.Series(dtype='int64')
for counts in per_file_labels.values():
    all_label_counts = all_label_counts.add(counts, fill_value=0)

all_label_counts = all_label_counts.astype(int).sort_values(ascending=False)
all_label_counts_df = pd.DataFrame({
    'Count': all_label_counts,
    'Percentage': (all_label_counts / all_label_counts.sum() * 100).round(4)
})
all_label_counts_df.index.name = 'Label'
print(f'Total samples: {all_label_counts.sum():,}\n')
all_label_counts_df

Total samples: 2,830,743



,Count,Percentage
Label,,
BENIGN,2273097,80.3004
DoS Hulk,231073,8.1630
PortScan,158930,5.6144
DDoS,128027,4.5227
DoS GoldenEye,10293,0.3636
FTP-Patator,7938,0.2804
SSH-Patator,5897,0.2083
DoS slowloris,5796,0.2048
DoS Slowhttptest,5499,0.1943


## Label-to-File Mapping

Which labels appear in which files?

In [7]:
label_file_map = {}
for filename, counts in per_file_labels.items():
    short_name = filename.replace('.pcap_ISCX.csv', '')
    for label in counts.index:
        if label not in label_file_map:
            label_file_map[label] = []
        label_file_map[label].append(short_name)

for label in all_label_counts.index:
    files = label_file_map[label]
    print(f'{label} ({len(files)} file{"s" if len(files) > 1 else ""}):')
    for f in files:
        print(f'  - {f}')

BENIGN (8 files):
  - Friday-WorkingHours-Afternoon-DDos
  - Friday-WorkingHours-Afternoon-PortScan
  - Friday-WorkingHours-Morning
  - Monday-WorkingHours
  - Thursday-WorkingHours-Afternoon-Infilteration
  - Thursday-WorkingHours-Morning-WebAttacks
  - Tuesday-WorkingHours
  - Wednesday-workingHours
DoS Hulk (1 file):
  - Wednesday-workingHours
PortScan (1 file):
  - Friday-WorkingHours-Afternoon-PortScan
DDoS (1 file):
  - Friday-WorkingHours-Afternoon-DDos
DoS GoldenEye (1 file):
  - Wednesday-workingHours
FTP-Patator (1 file):
  - Tuesday-WorkingHours
SSH-Patator (1 file):
  - Tuesday-WorkingHours
DoS slowloris (1 file):
  - Wednesday-workingHours
DoS Slowhttptest (1 file):
  - Wednesday-workingHours
Bot (1 file):
  - Friday-WorkingHours-Morning
Web Attack – Brute Force (1 file):
  - Thursday-WorkingHours-Morning-WebAttacks
Web Attack – XSS (1 file):
  - Thursday-WorkingHours-Morning-WebAttacks
Infiltration (1 file):
  - Thursday-WorkingHours-Afternoon-Infilteration
Web Attack – S

# 2. Duplicate Detection

## Within-File Duplicates

In [8]:
for filename, df in dfs.items():
    short_name = filename.replace('.pcap_ISCX.csv', '')
    n_dupes = df.duplicated().sum()
    pct = n_dupes / len(df) * 100
    print(f'{short_name}: {n_dupes:,} exact duplicates ({pct:.4f}%)')

Friday-WorkingHours-Afternoon-DDos: 2 exact duplicates (0.0009%)
Friday-WorkingHours-Afternoon-PortScan: 1 exact duplicates (0.0003%)
Friday-WorkingHours-Morning: 2 exact duplicates (0.0010%)
Monday-WorkingHours: 34 exact duplicates (0.0064%)
Thursday-WorkingHours-Afternoon-Infilteration: 142 exact duplicates (0.0492%)
Thursday-WorkingHours-Morning-WebAttacks: 288,602 exact duplicates (62.8806%)
Tuesday-WorkingHours: 4 exact duplicates (0.0009%)
Wednesday-workingHours: 17 exact duplicates (0.0025%)


## Cross-File Duplicates

Check if identical rows appear in multiple files. Uses the Flow ID, Source IP, Source Port, Destination IP, Destination Port, Timestamp columns as a composite key.

In [9]:
key_cols = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp']

common_key_cols = [c for c in key_cols if all(c in df.columns for df in dfs.values())]
print(f'Using key columns: {common_key_cols}\n')

if common_key_cols:
    tagged_keys = []
    for filename, df in dfs.items():
        short_name = filename.replace('.pcap_ISCX.csv', '')
        keys = df[common_key_cols].copy()
        keys['_source_file'] = short_name
        tagged_keys.append(keys)

    all_keys = pd.concat(tagged_keys, ignore_index=True)
    dupes = all_keys[all_keys.duplicated(subset=common_key_cols, keep=False)]
    n_cross_dupes = dupes.drop_duplicates(subset=common_key_cols).shape[0]
    print(f'Unique flow keys appearing in multiple rows: {n_cross_dupes:,} (out of {all_keys.shape[0]:,} total rows)')

    if n_cross_dupes > 0:
        cross_file = dupes.groupby(common_key_cols)['_source_file'].apply(lambda x: sorted(x.unique())).reset_index()
        cross_file_multi = cross_file[cross_file['_source_file'].apply(len) > 1]
        print(f'Keys appearing in multiple files: {len(cross_file_multi):,}')
        if len(cross_file_multi) > 0:
            print('\nSample cross-file duplicates:')
            display(cross_file_multi.head(10))
else:
    print('No common key columns found across files.')

Using key columns: ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp']

Unique flow keys appearing in multiple rows: 273,986 (out of 3,119,345 total rows)
Keys appearing in multiple files: 0


# 3. Missing Data Analysis

## Per-File Missing Values

In [10]:
for filename, df in dfs.items():
    short_name = filename.replace('.pcap_ISCX.csv', '')
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f'\n--- {short_name} ---')
        for col, count in missing.items():
            pct = count / len(df) * 100
            print(f'  {col}: {count:,} missing ({pct:.4f}%)')
    else:
        print(f'{short_name}: No missing values')


--- Friday-WorkingHours-Afternoon-DDos ---
  Flow Bytes/s: 4 missing (0.0018%)

--- Friday-WorkingHours-Afternoon-PortScan ---
  Flow Bytes/s: 15 missing (0.0052%)

--- Friday-WorkingHours-Morning ---
  Flow Bytes/s: 28 missing (0.0147%)

--- Monday-WorkingHours ---
  Flow Bytes/s: 64 missing (0.0121%)

--- Thursday-WorkingHours-Afternoon-Infilteration ---
  Flow Bytes/s: 18 missing (0.0062%)

--- Thursday-WorkingHours-Morning-WebAttacks ---
  Flow ID: 288,602 missing (62.8806%)
  Source IP: 288,602 missing (62.8806%)
  Source Port: 288,602 missing (62.8806%)
  Destination IP: 288,602 missing (62.8806%)
  Destination Port: 288,602 missing (62.8806%)
  Protocol: 288,602 missing (62.8806%)
  Timestamp: 288,602 missing (62.8806%)
  Flow Duration: 288,602 missing (62.8806%)
  Total Fwd Packets: 288,602 missing (62.8806%)
  Total Backward Packets: 288,602 missing (62.8806%)
  Total Length of Fwd Packets: 288,602 missing (62.8806%)
  Total Length of Bwd Packets: 288,602 missing (62.8806%)
 

## Infinity Values in Numeric Columns

Some columns like `Flow Bytes/s` may contain infinity values that aren't captured as NaN.

In [11]:
for filename, df in dfs.items():
    short_name = filename.replace('.pcap_ISCX.csv', '')
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = {}
    for col in numeric_cols:
        n_inf = np.isinf(df[col]).sum()
        if n_inf > 0:
            inf_counts[col] = n_inf
    if inf_counts:
        print(f'\n--- {short_name} ---')
        for col, count in inf_counts.items():
            print(f'  {col}: {count:,} infinite values')
    else:
        print(f'{short_name}: No infinite values')


--- Friday-WorkingHours-Afternoon-DDos ---
  Flow Bytes/s: 30 infinite values
  Flow Packets/s: 34 infinite values

--- Friday-WorkingHours-Afternoon-PortScan ---
  Flow Bytes/s: 356 infinite values
  Flow Packets/s: 371 infinite values

--- Friday-WorkingHours-Morning ---
  Flow Bytes/s: 94 infinite values
  Flow Packets/s: 122 infinite values

--- Monday-WorkingHours ---
  Flow Bytes/s: 373 infinite values
  Flow Packets/s: 437 infinite values

--- Thursday-WorkingHours-Afternoon-Infilteration ---
  Flow Bytes/s: 189 infinite values
  Flow Packets/s: 207 infinite values

--- Thursday-WorkingHours-Morning-WebAttacks ---
  Flow Bytes/s: 115 infinite values
  Flow Packets/s: 135 infinite values

--- Tuesday-WorkingHours ---
  Flow Bytes/s: 63 infinite values
  Flow Packets/s: 264 infinite values

--- Wednesday-workingHours ---
  Flow Bytes/s: 289 infinite values
  Flow Packets/s: 1,297 infinite values


## Aggregated Missing Data Summary

In [12]:
all_missing = pd.DataFrame()
for filename, df in dfs.items():
    short_name = filename.replace('.pcap_ISCX.csv', '')
    missing = df.isnull().sum()
    missing.name = short_name
    all_missing = pd.concat([all_missing, missing.to_frame().T])

cols_with_missing = all_missing.columns[all_missing.sum() > 0]
if len(cols_with_missing) > 0:
    print('Columns with missing values (count per file):\n')
    display(all_missing[cols_with_missing])
else:
    print('No columns have missing values in any file.')

Columns with missing values (count per file):



,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
Friday-WorkingHours-Afternoon-DDos,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Friday-WorkingHours-Afternoon-PortScan,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Friday-WorkingHours-Morning,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,28,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Monday-WorkingHours,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,64,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Thursday-WorkingHours-Afternoon-Infilteration,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Thursday-WorkingHours-Morning-WebAttacks,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288622,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602,288602
Tuesday-WorkingHours,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,201,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Wednesday-workingHours,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1008,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# 4. Skewness Analysis

We concatenate all files and compute skewness on numeric columns. Skewness > 2 or < -2 indicates strong skew.

In [13]:
all_dfs = []
for filename, df in dfs.items():
    tagged = df.copy()
    tagged['_source_file'] = filename.replace('.pcap_ISCX.csv', '')
    all_dfs.append(tagged)

combined_df = pd.concat(all_dfs, ignore_index=True)
print(f'Combined dataset: {combined_df.shape[0]:,} rows, {combined_df.shape[1]} columns')

Combined dataset: 3,119,345 rows, 86 columns


In [14]:
numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
print(f'{len(numeric_cols)} numeric columns\n')

numeric_data = combined_df[numeric_cols].replace([np.inf, -np.inf], np.nan)

skewness = numeric_data.skew().sort_values(key=abs, ascending=False)
skew_df = pd.DataFrame({
    'Skewness': skewness,
    'Abs Skewness': skewness.abs(),
    'Assessment': skewness.apply(lambda x: 
        'Highly right-skewed' if x > 2 else
        'Moderately right-skewed' if x > 1 else
        'Highly left-skewed' if x < -2 else
        'Moderately left-skewed' if x < -1 else
        'Approximately symmetric'
    )
})
skew_df

80 numeric columns



,Skewness,Abs Skewness,Assessment
Fwd Header Length.1,"-1,325.0738","1,325.0738",Highly left-skewed
Fwd Header Length,"-1,325.0738","1,325.0738",Highly left-skewed
Total Length of Fwd Packets,805.5705,805.5705,Highly right-skewed
Subflow Fwd Bytes,803.5986,803.5986,Highly right-skewed
Bwd Header Length,-717.0313,717.0313,Highly left-skewed
min_seg_size_forward,-474.5353,474.5353,Highly left-skewed
act_data_pkt_fwd,284.5952,284.5952,Highly right-skewed
Total Backward Packets,244.6795,244.6795,Highly right-skewed
Subflow Bwd Packets,244.6795,244.6795,Highly right-skewed
Total Fwd Packets,244.3806,244.3806,Highly right-skewed


In [15]:
highly_skewed = skew_df[skew_df['Abs Skewness'] > 2]
print(f'{len(highly_skewed)} columns with |skewness| > 2 (strongly skewed):')
print(f'{len(skew_df[skew_df["Abs Skewness"] > 1])} columns with |skewness| > 1 (moderately+ skewed):')
print(f'{len(skew_df[skew_df["Abs Skewness"] <= 1])} columns with |skewness| <= 1 (approximately symmetric):')

68 columns with |skewness| > 2 (strongly skewed):
68 columns with |skewness| > 1 (moderately+ skewed):
12 columns with |skewness| <= 1 (approximately symmetric):


## Per-Class Skewness

Check if skewness patterns differ between attack types and benign traffic.

In [16]:
top_labels = combined_df['Label'].value_counts().head(5).index.tolist()

interesting_cols = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Flow Bytes/s', 'Flow Packets/s', 'Fwd Packet Length Mean',
    'Bwd Packet Length Mean', 'Flow IAT Mean', 'Packet Length Mean',
    'Average Packet Size'
]
interesting_cols = [c for c in interesting_cols if c in numeric_cols]

class_skew = {}
for label in top_labels:
    subset = combined_df.loc[combined_df['Label'] == label, interesting_cols]
    subset = subset.replace([np.inf, -np.inf], np.nan)
    class_skew[label] = subset.skew()

class_skew_df = pd.DataFrame(class_skew).round(2)
class_skew_df

,BENIGN,DoS Hulk,PortScan,DDoS,DoS GoldenEye
Flow Duration,2.7800,-0.3800,35.0700,1.7900,2.1800
Total Fwd Packets,219.0300,0.0600,272.1200,1.1300,0.2900
Total Backward Packets,219.2800,-0.5800,70.6600,-0.3500,-0.5700
Flow Bytes/s,42.6700,17.3900,11.7300,177.8600,0.9100
Flow Packets/s,5.7200,2.9800,11.5900,145.1000,17.8800
Fwd Packet Length Mean,8.2400,0.9300,26.3000,0.0300,1.7700
Bwd Packet Length Mean,3.3900,-0.5000,43.6700,-0.3300,-0.0400
Flow IAT Mean,9.7400,0.3200,35.4400,1.5000,2.2100
Packet Length Mean,3.7700,-0.5000,43.5900,-0.2700,-0.2800
Average Packet Size,3.8900,-0.4500,43.6500,-0.2200,-0.2900


# 5. Outlier Detection

Using the IQR method: values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR are flagged as outliers.

In [17]:
numeric_data = combined_df[numeric_cols].replace([np.inf, -np.inf], np.nan)

Q1 = numeric_data.quantile(0.25)
Q3 = numeric_data.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_counts = ((numeric_data < lower_bound) | (numeric_data > upper_bound)).sum()
outlier_pct = (outlier_counts / numeric_data.count() * 100)

outlier_df = pd.DataFrame({
    'Outlier Count': outlier_counts,
    'Outlier %': outlier_pct.round(4),
    'Q1': Q1.round(4),
    'Q3': Q3.round(4),
    'IQR': IQR.round(4),
    'Lower Bound': lower_bound.round(4),
    'Upper Bound': upper_bound.round(4),
    'Min': numeric_data.min().round(4),
    'Max': numeric_data.max().round(4),
}).sort_values('Outlier %', ascending=False)

outlier_df

,Outlier Count,Outlier %,Q1,Q3,IQR,Lower Bound,Upper Bound,Min,Max
Fwd IAT Mean,671833,23.7335,0.0000,"206,306.3584","206,306.3584","-309,459.5375","515,765.8959",0.0000,"120,000,000.0000"
Fwd IAT Max,666292,23.5377,0.0000,"931,006.0000","931,006.0000","-1,396,509.0000","2,327,515.0000",0.0000,"120,000,000.0000"
Fwd IAT Total,665766,23.5191,0.0000,"1,242,843.5000","1,242,843.5000","-1,864,265.2500","3,107,108.7500",0.0000,"120,000,000.0000"
Fwd Packet Length Max,664214,23.4643,6.0000,81.0000,75.0000,-106.5000,193.5000,0.0000,"24,820.0000"
Fwd Packet Length Std,663959,23.4553,0.0000,26.1630,26.1630,-39.2444,65.4074,0.0000,"7,125.5968"
Fwd IAT Std,660519,23.3338,0.0000,"65,989.8217","65,989.8217","-98,984.7326","164,974.5544",0.0000,"84,602,929.2770"
Packet Length Variance,654542,23.1226,0.0000,"30,388.8393","30,388.8393","-45,583.2590","75,972.0983",0.0000,"22,400,000.0000"
Bwd Packet Length Std,654269,23.1130,0.0000,77.9405,77.9405,-116.9108,194.8513,0.0000,"8,194.6605"
Bwd Packet Length Max,637112,22.5069,0.0000,280.0000,280.0000,-420.0000,700.0000,0.0000,"19,530.0000"
Destination Port,627245,22.1583,53.0000,443.0000,390.0000,-532.0000,"1,028.0000",0.0000,"65,535.0000"


In [18]:
high_outlier_cols = outlier_df[outlier_df['Outlier %'] > 10]
print(f'Columns with >10% outliers: {len(high_outlier_cols)}')
print()
for col in high_outlier_cols.index:
    print(f'  {col}: {high_outlier_cols.loc[col, "Outlier %"]:.4f}% outliers')

print(f'\nColumns with >5% outliers: {len(outlier_df[outlier_df["Outlier %"] > 5])}')
print(f'Columns with >1% outliers: {len(outlier_df[outlier_df["Outlier %"] > 1])}')
print(f'Columns with 0 outliers: {len(outlier_df[outlier_df["Outlier Count"] == 0])}')

Columns with >10% outliers: 50

  Fwd IAT Mean: 23.7335% outliers
  Fwd IAT Max: 23.5377% outliers
  Fwd IAT Total: 23.5191% outliers
  Fwd Packet Length Max: 23.4643% outliers
  Fwd Packet Length Std: 23.4553% outliers
  Fwd IAT Std: 23.3338% outliers
  Packet Length Variance: 23.1226% outliers
  Bwd Packet Length Std: 23.1130% outliers
  Bwd Packet Length Max: 22.5069% outliers
  Destination Port: 22.1583% outliers
  Max Packet Length: 22.0402% outliers
  Total Length of Bwd Packets: 21.8687% outliers
  Subflow Bwd Bytes: 21.8687% outliers
  Flow IAT Std: 21.1785% outliers
  Bwd IAT Std: 21.1766% outliers
  Flow IAT Mean: 20.2610% outliers
  Bwd IAT Max: 20.0867% outliers
  Idle Max: 20.0401% outliers
  Idle Min: 20.0401% outliers
  Idle Mean: 20.0401% outliers
  Bwd IAT Total: 19.9871% outliers
  Bwd IAT Mean: 19.9209% outliers
  Packet Length Std: 19.7729% outliers
  Active Min: 19.7414% outliers
  Active Max: 19.7414% outliers
  Active Mean: 19.7414% outliers
  Fwd IAT Min: 18.901

## Outliers by Class

Do certain attack types produce more outliers than benign traffic?

In [19]:
outlier_by_label = {}
for label in combined_df['Label'].unique():
    subset = combined_df.loc[combined_df['Label'] == label, numeric_cols].replace([np.inf, -np.inf], np.nan)
    n_outliers = ((subset < lower_bound) | (subset > upper_bound)).sum().sum()
    n_values = subset.count().sum()
    outlier_by_label[label] = {
        'Rows': len(subset),
        'Total Outlier Values': n_outliers,
        'Total Numeric Values': n_values,
        'Outlier Rate %': round(n_outliers / n_values * 100, 4) if n_values > 0 else 0
    }

outlier_by_label_df = pd.DataFrame(outlier_by_label).T.sort_values('Outlier Rate %', ascending=False)
outlier_by_label_df.index.name = 'Label'
outlier_by_label_df['Rows'] = outlier_by_label_df['Rows'].astype(int)
outlier_by_label_df['Total Outlier Values'] = outlier_by_label_df['Total Outlier Values'].astype(int)
outlier_by_label_df['Total Numeric Values'] = outlier_by_label_df['Total Numeric Values'].astype(int)
outlier_by_label_df

,Rows,Total Outlier Values,Total Numeric Values,Outlier Rate %
Label,,,,
Infiltration,36,1105,2880,38.3681
Heartbleed,11,336,880,38.1818
DoS GoldenEye,10293,272050,823440,33.0382
DoS Hulk,231073,4756589,18483942,25.7336
DoS Slowhttptest,5499,109005,439920,24.7784
DoS slowloris,5796,103122,463680,22.2399
DDoS,128027,1996016,10242156,19.4882
SSH-Patator,5897,80641,471760,17.0936
Web Attack – Sql Injection,21,258,1680,15.3571


## Extreme Values

Columns where the max value is orders of magnitude beyond the 99th percentile.

In [20]:
p99 = numeric_data.quantile(0.99)
p01 = numeric_data.quantile(0.01)
col_max = numeric_data.max()
col_min = numeric_data.min()
col_median = numeric_data.median()

extreme_cols = []
for col in numeric_cols:
    if p99[col] != 0 and col_max[col] / p99[col] > 10:
        extreme_cols.append({
            'Column': col,
            'Median': col_median[col],
            'P99': p99[col],
            'Max': col_max[col],
            'Max/P99 Ratio': col_max[col] / p99[col]
        })
    elif p01[col] != 0 and col_min[col] / p01[col] > 10:
        extreme_cols.append({
            'Column': col,
            'Median': col_median[col],
            'P01': p01[col],
            'Min': col_min[col],
            'Min/P01 Ratio': col_min[col] / p01[col]
        })

if extreme_cols:
    extreme_df = pd.DataFrame(extreme_cols).set_index('Column').sort_values('Max/P99 Ratio', ascending=False, na_position='last')
    print(f'{len(extreme_cols)} columns where max is >10x the 99th percentile:\n')
    display(extreme_df)
else:
    print('No columns with extreme outliers (max > 10x P99).')

23 columns where max is >10x the 99th percentile:



,Median,P99,Max,Max/P99 Ratio
Column,,,,
Total Length of Bwd Packets,123.0000,"71,831.0000","655,453,030.0000","9,124.9326"
Subflow Bwd Bytes,123.0000,"71,831.0000","655,453,030.0000","9,124.9326"
act_data_pkt_fwd,1.0000,27.0000,"213,557.0000","7,909.5185"
Subflow Bwd Packets,2.0000,57.0000,"291,922.0000","5,121.4386"
Total Backward Packets,2.0000,57.0000,"291,922.0000","5,121.4386"
Total Fwd Packets,2.0000,48.0000,"219,759.0000","4,578.3125"
Subflow Fwd Packets,2.0000,48.0000,"219,759.0000","4,578.3125"
Bwd Header Length,40.0000,"1,520.0000","5,838,440.0000","3,841.0789"
Fwd Header Length,64.0000,"1,328.0000","4,644,908.0000","3,497.6717"


# 6. Correlation Analysis

Compute pairwise Pearson correlations among all numeric columns and identify strongly correlated feature pairs.

In [21]:
corr_matrix = numeric_data.corr()

# Extract upper triangle (no diagonal) to avoid duplicate pairs
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Collect all correlated pairs with |r| > 0.8
strong_threshold = 0.8
strong_pairs = []
for col in upper_tri.columns:
    for idx in upper_tri.index:
        val = upper_tri.loc[idx, col]
        if pd.notna(val) and abs(val) > strong_threshold:
            strong_pairs.append({
                'Feature 1': idx,
                'Feature 2': col,
                'Correlation': round(val, 4),
                'Abs Correlation': round(abs(val), 4)
            })

strong_corr_df = pd.DataFrame(strong_pairs).sort_values('Abs Correlation', ascending=False)

n_total_pairs = upper_tri.count().sum()
n_strong = len(strong_corr_df)
n_very_strong = len(strong_corr_df[strong_corr_df['Abs Correlation'] > 0.95])
n_perfect = len(strong_corr_df[strong_corr_df['Abs Correlation'] > 0.99])

print(f'Total feature pairs evaluated: {n_total_pairs:,}')
print(f'Strong correlations (|r| > 0.80): {n_strong}')
print(f'Very strong correlations (|r| > 0.95): {n_very_strong}')
print(f'Near-perfect correlations (|r| > 0.99): {n_perfect}')
print(f'\nTop 30 strongest correlated pairs:\n')
strong_corr_df.head(30)

Total feature pairs evaluated: 2,556
Strong correlations (|r| > 0.80): 102
Very strong correlations (|r| > 0.95): 39
Near-perfect correlations (|r| > 0.99): 27

Top 30 strongest correlated pairs:



,Feature 1,Feature 2,Correlation,Abs Correlation
45,Fwd PSH Flags,SYN Flag Count,1.0000,1.0000
76,Total Length of Bwd Packets,Subflow Bwd Bytes,1.0000,1.0000
46,Fwd URG Flags,CWE Flag Count,1.0000,1.0000
56,Fwd Packet Length Mean,Avg Fwd Segment Size,1.0000,1.0000
59,Bwd Packet Length Mean,Avg Bwd Segment Size,1.0000,1.0000
65,Fwd Header Length,Fwd Header Length.1,1.0000,1.0000
66,Total Fwd Packets,Subflow Fwd Packets,1.0000,1.0000
69,Total Length of Fwd Packets,Subflow Fwd Bytes,1.0000,1.0000
71,Total Backward Packets,Subflow Bwd Packets,1.0000,1.0000
67,Total Backward Packets,Subflow Fwd Packets,0.9991,0.9991


## Correlation Distribution

Breakdown of all pairwise correlations by strength.

In [22]:
# Flatten upper triangle into a series of correlation values
all_corrs = upper_tri.stack().abs()

bins = [0, 0.2, 0.4, 0.6, 0.8, 0.95, 1.0]
labels = ['Very weak (0-0.2)', 'Weak (0.2-0.4)', 'Moderate (0.4-0.6)',
          'Strong (0.6-0.8)', 'Very strong (0.8-0.95)', 'Near-perfect (0.95-1.0)']
corr_bins = pd.cut(all_corrs, bins=bins, labels=labels, include_lowest=True)
corr_dist = corr_bins.value_counts().reindex(labels)

corr_dist_df = pd.DataFrame({
    'Count': corr_dist,
    'Percentage': (corr_dist / corr_dist.sum() * 100).round(2)
})
corr_dist_df.index.name = 'Correlation Strength'
corr_dist_df

,Count,Percentage
Correlation Strength,,
Very weak (0-0.2),2053,80.3200
Weak (0.2-0.4),220,8.6100
Moderate (0.4-0.6),132,5.1600
Strong (0.6-0.8),49,1.9200
Very strong (0.8-0.95),63,2.4600
Near-perfect (0.95-1.0),39,1.5300


# 7. Summary

In [23]:
print('=' * 70)
print('DATASET SUMMARY')
print('=' * 70)
print(f'\nFiles: {len(csv_files)}')
print(f'Total rows: {combined_df.shape[0]:,}')
print(f'Columns: {combined_df.shape[1] - 1} (excluding _source_file tag)')
print(f'\n--- Class Distribution ---')
print(f'Unique labels: {combined_df["Label"].nunique()}')
for label, count in all_label_counts.items():
    pct = count / all_label_counts.sum() * 100
    print(f'  {label}: {count:,} ({pct:.4f}%)')
print(f'\n--- Data Quality ---')
total_missing = combined_df[numeric_cols].isnull().sum().sum()
total_inf = sum(np.isinf(combined_df[col]).sum() for col in numeric_cols if combined_df[col].dtype in [np.float64, np.float32])
total_dupes = combined_df.drop(columns=['_source_file']).duplicated().sum()
print(f'Total missing values: {total_missing:,}')
print(f'Total infinite values: {total_inf:,}')
print(f'Total exact duplicate rows: {total_dupes:,} ({total_dupes/len(combined_df)*100:.4f}%)')
print(f'\n--- Skewness ---')
print(f'Strongly skewed columns (|skew| > 2): {len(skew_df[skew_df["Abs Skewness"] > 2])}')
print(f'Approximately symmetric (|skew| <= 1): {len(skew_df[skew_df["Abs Skewness"] <= 1])}')
print(f'\n--- Outliers (IQR method) ---')
print(f'Columns with >10% outliers: {len(outlier_df[outlier_df["Outlier %"] > 10])}')
print(f'Columns with >5% outliers: {len(outlier_df[outlier_df["Outlier %"] > 5])}')
print(f'Columns with 0 outliers: {len(outlier_df[outlier_df["Outlier Count"] == 0])}')
print(f'\n--- Correlations ---')
print(f'Total feature pairs evaluated: {n_total_pairs:,}')
print(f'Strong correlations (|r| > 0.80): {n_strong}')
print(f'Very strong correlations (|r| > 0.95): {n_very_strong}')
print(f'Near-perfect correlations (|r| > 0.99): {n_perfect}')

DATASET SUMMARY

Files: 8
Total rows: 3,119,345
Columns: 85 (excluding _source_file tag)

--- Class Distribution ---
Unique labels: 15
  BENIGN: 2,273,097 (80.3004%)
  DoS Hulk: 231,073 (8.1630%)
  PortScan: 158,930 (5.6144%)
  DDoS: 128,027 (4.5227%)
  DoS GoldenEye: 10,293 (0.3636%)
  FTP-Patator: 7,938 (0.2804%)
  SSH-Patator: 5,897 (0.2083%)
  DoS slowloris: 5,796 (0.2048%)
  DoS Slowhttptest: 5,499 (0.1943%)
  Bot: 1,966 (0.0695%)
  Web Attack – Brute Force: 1,507 (0.0532%)
  Web Attack – XSS: 652 (0.0230%)
  Infiltration: 36 (0.0013%)
  Web Attack – Sql Injection: 21 (0.0007%)
  Heartbleed: 11 (0.0004%)

--- Data Quality ---
Total missing values: 23,089,518
Total infinite values: 4,376
Total exact duplicate rows: 288,804 (9.2585%)

--- Skewness ---
Strongly skewed columns (|skew| > 2): 68
Approximately symmetric (|skew| <= 1): 12

--- Outliers (IQR method) ---
Columns with >10% outliers: 50
Columns with >5% outliers: 56
Columns with 0 outliers: 12

--- Correlations ---
Total feat